# DATA MINING EN ECONOMIA Y FINANZAS 2026
  Comisión jueves

# El Arbol de Marga Kotmilo



### Seteo del ambiente en Google Colab

Esta parte se debe correr con el runtime en Python3
<br>Ir al menu, Runtime -> Change Runtime Type -> Runtime type ->  **Python 3**

Conectar la virtual machine donde está corriendo Google Colab con el  Google Drive, para poder tener persistencia de archivos.

In [1]:
# primero establecer el Runtime de Python 3
from google.colab import drive
drive.mount('/content/.drive')

Mounted at /content/.drive


los siguientes comandos estan en shell script de Linux

* Crear las carpetas en el Google Drive
* Bajar el competencia_01_crudo al Google Drive y tambien al disco local de la virtual machine que está corriendo Google Colab



In [2]:
%%shell

mkdir -p "/content/.drive/My Drive/dmeyf"
mkdir -p "/content/buckets"
ln -sfn "/content/.drive/My Drive/dmeyf" /content/buckets/b1


mkdir -p /content/buckets/b1/exp
mkdir -p /content/buckets/b1/datasets
mkdir -p /content/datasets


# defino funcion descargar()
descargar() {
  carpeta_destino="/content/buckets/b1/datasets/"
  url_origen="https://storage.googleapis.com/open-courses/dmeyf2026-9c6f/"
  archivo="$1"

  if ! test -f "$carpeta_destino""$archivo"; then
    wget  "$url_origen""$archivo"  -O "$carpeta_destino""$archivo"
  fi

  if ! test -f  "/content/datasets/""$archivo"; then
    cp  "$carpeta_destino""$archivo"  "/content/datasets/""$archivo"
  fi;
}


# hago la descarga efectiva, llamando a descargar()
descargar  "competencia_01_crudo.csv"

## Generacion de la clase_ternaria

Esta parte se debe correr con el runtime en lenguaje **R** Ir al menu, Runtime -> Change Runtime Type -> Runtime type -> R

In [ ]:
require( "data.table" )

# leo el dataset
dataset <- fread("/content/datasets/competencia_01_crudo.csv" )

# calculo el periodo0 consecutivo
dsimple <- dataset[, list(
    "pos" = .I,
    numero_de_cliente,
    periodo0 = as.integer(foto_mes/100)*12 +  foto_mes%%100 ) ]


# ordeno
setorder( dsimple, numero_de_cliente, periodo0 )

# calculo topes
periodo_ultimo <- dsimple[, max(periodo0) ]
periodo_anteultimo <- periodo_ultimo - 1


# calculo los leads de orden 1 y 2
dsimple[, c("periodo1", "periodo2") :=
    shift(periodo0, n=1:2, fill=NA, type="lead"),  numero_de_cliente ]

# assign most common class values = "CONTINUA"
dsimple[ periodo0 < periodo_anteultimo, clase_ternaria := "CONTINUA" ]

# calculo BAJA+1
dsimple[ periodo0 < periodo_ultimo &
    ( is.na(periodo1) | periodo0 + 1 < periodo1 ),
    clase_ternaria := "BAJA+1" ]

# calculo BAJA+2
dsimple[ periodo0 < periodo_anteultimo & (periodo0+1 == periodo1 )
    & ( is.na(periodo2) | periodo0 + 2 < periodo2 ),
    clase_ternaria := "BAJA+2" ]


# pego el resultado en el dataset original y grabo
setorder( dsimple, pos )
dataset[, clase_ternaria := dsimple$clase_ternaria ]

fwrite( dataset,
    file =  "/content/datasets/competencia_01.csv.gz",
    sep = ","
)

In [ ]:
setorder( dataset, foto_mes, clase_ternaria, numero_de_cliente)
dataset[, .N, list(foto_mes, clase_ternaria)]

## El Arbol de Marga

limpio el ambiente de R

In [ ]:
# limpio la memoria
rm(list=ls(all.names=TRUE)) # remove all objects
gc(full=TRUE, verbose=FALSE) # garbage collection

In [ ]:
# cargo las librerias que necesito
require("data.table")
require("rpart")
require("parallel")

if(!require("rpart.plot")) install.packages("rpart.plot")
require("rpart.plot")

if(!require("R.utils")) install.packages("R.utils")
require("R.utils")

if (!require("primes")) install.packages("primes")
require("primes")

In [ ]:
PARAM <- list()
PARAM$experimento <- "marga0312"
PARAM$semilla_primigenia <- 265621 # reemplazar por su primer semilla
PARAM$qsemillas <- 3
PARAM$peso_baja2 <- 1.0

PARAM$training_pct <- 70L  # entre  1L y 99L

a1 <- list(
  "cp"= -1,
  "maxdepth"= 7,
  "minsplit"= 1500,
  "minbucket"= 500
)

a2 <- list(
  "cp"= -1,
  "maxdepth"= 9,
  "minsplit"= 1000,
  "minbucket"= 300
)

PARAM$param_basicos <- list( a1, a2 )

In [ ]:
# genero numeros primos
primos <- generate_primes(min = 100000, max = 1000000)
set.seed(PARAM$semilla_primigenia) # inicializo
# me quedo con PARAM$qsemillas   semillas
PARAM$semillas <- sample(primos, PARAM$qsemillas )

PARAM$semillas

In [ ]:
# Carpeta del experimento
setwd("/content/buckets/b1/exp")
dir.create(PARAM$experimento, showWarnings=FALSE)
setwd( paste0("/content/buckets/b1/exp/", PARAM$experimento ))

In [ ]:
# particionar agrega una columna llamada fold a un dataset
#  que consiste en una particion estratificada segun agrupa
# particionar( data=dataset, division=c(70,30), agrupa=clase_ternaria, seed=semilla)
#   crea una particion 70, 30

particionar <- function(data, division, agrupa = "", campo = "fold", start = 1, seed = NA) {
  if (!is.na(seed)) set.seed(seed)

  bloque <- unlist(mapply(function(x, y) {
    rep(y, x)
  }, division, seq(from = start, length.out = length(division))))

  data[, (campo) := sample(rep(bloque, ceiling(.N / length(bloque))))[1:.N],
    by = agrupa
  ]
}


In [ ]:
# lectura del dataset
dataset <- fread("/content/datasets/competencia_01.csv.gz")

# trabajo solo con los datos de 202106, ultimo mes con clase_ternaria completa
# filtro datos
dataset <- dataset[foto_mes==202106]

invisible(gc(full=TRUE, verbose=FALSE)) # garbage collection

nrow(dataset)
dataset[, .N, clase_ternaria]

In [ ]:
if(file.exists("tb_marga_detalle.txt")){
  tb_marga_detalle <- fread("tb_marga_detalle.txt")
}else{
  tb_marga_detalle <- data.table(
    corrida= integer(),
    semilla= integer(),
    peso_baja2= numeric(),
    cp= numeric(),
    maxdepth= integer(),
    minsplit= integer(),
    minbucket= integer(),
    ganancia_test= numeric(),
    profundidad_real= numeric(),
    minsplit_real= numeric(),
    minbucket_real= numeric(),
    hojas_cantidad= numeric(),
    complexity_raiz= numeric(),
    complexity_min= numeric()
  )
}

if( 0 == nrow( tb_marga_detalle ) ) {
  corrida <- 1
} else {
  corrida <- 1 + tb_marga_detalle[, max(corrida)]
}


In [ ]:
# visualizo que tiene la tabla
tb_marga_detalle

In [ ]:
# genero para cada semilla un arbol,  demoraaaa


for( semilla in PARAM$semillas ) {
for( iarbol in seq(length( PARAM$param_basicos))) {


  # particiono estratificadamente el dataset
  particionar(dataset,
    division = c(PARAM$training_pct, 100L -PARAM$training_pct),
    agrupa = "clase_ternaria",
    seed = semilla # aqui se usa SU semilla
  )

  # genero el modelo
  # predecir clase_ternaria a partir del resto
  pesos <- dataset[fold == 1, ifelse( clase_ternaria=="BAJA+2", PARAM$peso_baja2, 1.0 ) ]

  modelo <- rpart("clase_ternaria ~ .",
    data = dataset[fold == 1], # fold==1  es training,  el 70% de los datos
    xval = 0,
    control = PARAM$param_basicos[[iarbol]],
    weights= pesos
  ) # aqui van los parametros del arbol

  # impresion del arbol en un pdf
  arch_arbol <- paste0( "arbol_", corrida, "_", semilla, ".pdf")
  pdf(file = arch_arbol, width=28, height=4)
  prp(modelo, extra=101, digits=5, branch=1, type=4, varlen=0, faclen=0)
  dev.off()

  # Genero el arbol como tabla
  tb_arbol_tabla <- as.data.table(modelo$frame, keep.rownames = "nodo")
  arch_tabla <- paste0( "arbol_", corrida, "_", semilla, ".txt")
  fwrite(tb_arbol_tabla,
    file= arch_tabla,
    sep="\t"
  )

  # calculo de metricas
  complexity_min <- tb_arbol_tabla[ var!="<leaf>" , min(complexity) ]

  # aplico el modelo a los datos de testing
  prediccion <- predict(modelo, # el modelo que genere recien
    dataset[fold == 2], # fold==2  es testing, el 30% de los datos
    type = "prob"
  ) # type= "prob"  es que devuelva la probabilidad

  # prediccion es una matriz con TRES columnas,
  #  llamadas "BAJA+1", "BAJA+2"  y "CONTINUA"
  # cada columna es el vector de probabilidades


  # calculo la ganancia en testing  qu es fold==2
  ganancia_test <- dataset[
    fold == 2,
    sum(ifelse(prediccion[, "BAJA+2"] >  (PARAM$peso_baja2 / ( PARAM$peso_baja2 + 39.0 ) ),
      ifelse(clase_ternaria == "BAJA+2", 1072500, -27500),
      0
    ))
  ]

  # escalo la ganancia como si fuera todo el dataset
  ganancia_test_normalizada <- ganancia_test / (( 100 - PARAM$training_pct ) / 100 )

  resultado <-
    c( list("corrida"=corrida, "semilla"= semilla, "peso_baja2"=PARAM$peso_baja2),
      PARAM$param_basicos[[iarbol]],
      list( "ganancia_test"= ganancia_test_normalizada,
       "profundidad_real"= max( rpart:::tree.depth( as.numeric(rownames(modelo$frame)) ) ),
       "minsplit_real"= tb_arbol_tabla[ var!="<leaf>" , min( yval2.V2 + yval2.V3/PARAM$peso_baja2 + yval2.V4) ],
       "minbucket_real"= tb_arbol_tabla[ var=="<leaf>" , min( yval2.V2 + yval2.V3/PARAM$peso_baja2 + yval2.V4) ],
       "hojas_cantidad"= tb_arbol_tabla[ var=="<leaf>" , .N ],
       "complexity_raiz"=  tb_arbol_tabla[ nodo==1, complexity],
       "complexity_min"= tb_arbol_tabla[ var!="<leaf>" , min(complexity) ]
       )
     )

  # agrego a la tabla
  tb_marga_detalle <- rbindlist(list( tb_marga_detalle, resultado ))
  fwrite( tb_marga_detalle, "tb_marga_detalle.txt", sep="\t")

}
}


In [ ]:
# la tabla de las corridas
tb_marga_detalle

In [ ]:
# la impresion del primer arbol en formato tabla
getwd()
tb_arbol <- fread( paste0( "arbol_", corrida, "_", PARAM$semillas[1], ".txt"))
tb_arbol